<a href="https://colab.research.google.com/github/sangpm2005/TTNT/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import copy
import math
import random
import numpy

# Ký hiệu người chơi
X = "X"
O = "O"
EMPTY = None

# Biến toàn cục xác định người và máy
user = None
ai = None


# Trả về trạng thái bàn cờ ban đầu 3x3 gồm toàn EMPTY
def initial_state():
  return [[EMPTY, EMPTY, EMPTY],
          [EMPTY, EMPTY, EMPTY],
          [EMPTY, EMPTY, EMPTY]]

# Xác định lượt chơi dựa trên số ô đã đánh
def player(board):
  count = 0
  # Đếm số ô không rỗng
  for i in board:
    for j in i:
      if j:
        count += 1
  # Nếu số lượng nước đi là lẻ → đến lượt ai
  if count % 2 != 0:
    return ai
  return user

# Trả về tập hợp các ô còn trống
def actions(board):
  res = set()
  board_len = len(board)
  for i in range(board_len):
    for j in range(board_len):
      if board[i][j] == EMPTY:
        res.add((i, j))
  return res

# Trả về một cấu hình mới sau khi đặt một nước đi
def result(board, action):
  curr_player = player(board)          # Xác định ai đang đánh
  result_board = copy.deepcopy(board)  # Sao chép bàn cờ tránh ảnh hưởng bản gốc
  (i, j) = action
  result_board[i][j] = curr_player     # Đặt quân
  return result_board

# Kiểm tra thắng theo hàng
def get_horizontal_winner(board):
  winner_val = None
  board_len = len(board)
  for i in range(board_len):
    winner_val = board[i][0]           # Lấy giá trị ô đầu tiên của hàng
    for j in range(board_len):
      if board[i][j] != winner_val:
        winner_val = None              # Nếu có ô khác → không thắng hàng này
    if winner_val:
      return winner_val                # Trả về X hoặc O
  return winner_val

# Kiểm tra thắng theo cột
def get_vertical_winner(board):
  winner_val = None
  board_len = len(board)
  for i in range(board_len):
    winner_val = board[0][i]
    for j in range(board_len):
      if board[j][i] != winner_val:
        winner_val = None
      if winner_val:
        return winner_val
  return winner_val

# Kiểm tra thắng theo đường chéo
def get_diagonal_winner(board):
  winner_val = None
  board_len = len(board)

  # Đường chéo chính
  winner_val = board[0][0]
  for i in range(board_len):
    if board[i][i] != winner_val:
      winner_val = None
  if winner_val:
    return winner_val

  # Đường chéo phụ
  winner_val = board[0][board_len - 1]
  for i in range(board_len):
      j = board_len - 1 - i
      if board[i][j] != winner_val:
        winner_val = None
  return winner_val

# Trả về người chiến thắng nếu có
def winner(board):
  winner_val = get_horizontal_winner(board) or get_vertical_winner(board) or get_diagonal_winner(board) or None
  return winner_val

# Trạng thái kết thúc: thắng hoặc hòa (không còn ô trống)
def terminal(board):
  if winner(board) != None:
    return True
  for i in board:
    for j in i:
      if j == EMPTY:
        return False
  return True

# Giá trị kết thúc cho minimax
def utility(board):
  winner_val = winner(board)
  if winner_val == X:
    return 1
  elif winner_val == O:
    return -1
  return 0

# Player Maximizing (X)
def maxValue(state):
  if terminal(state):
    return utility(state)
  v = -math.inf
  for action in actions(state):
    v = max(v, minValue(result(state, action)))  # Gọi phía min
  return v

# Player Minimizing (O)
def minValue(state):
  if terminal(state):
    return utility(state)
  v = math.inf
  for action in actions(state):
    v = min(v, maxValue(result(state, action)))
  return v

# Minimax chọn nước đi tốt nhất cho AI
def minimax(board):
  current_player = player(board)
  if current_player == X:
    min = -math.inf
    for action in actions(board):
      check = minValue(result(board, action))
      if check > min:
        min = check
        move = action
  else:
    max = math.inf
    for action in actions(board):
      check = maxValue(result(board, action))
      if check < max:
        max = check
        move = action
  return move

# Chương trình chính
if __name__ == "__main__":
  board = initial_state()
  ai_turn = False

  print("Choose a player")
  user = input()       # Người chọn X hoặc O

  # Gán AI = quân còn lại
  if user == "X":
    ai = "O"
  else:
    ai = "X"

  # Vòng lặp chơi game
  while True:
    game_over = terminal(board)   # Kiểm tra kết thúc
    playr = player(board)         # Xác định người chơi hiện tại

    if game_over:
      winner = winner(board)
      if winner is None:
        print("Game Over: Tie.")
      else:
        print(f"Game Over: {winner} wins.")
      break

    else:
      # Lượt AI
      if user != playr and not game_over:
        if ai_turn:
          move = minimax(board)       # AI chọn nước đi tốt nhất
          board = result(board, move)
          ai_turn = False
          print(numpy.array(board))

      # Lượt người chơi
      elif user == playr and not game_over:
        ai_turn = True
        print("Enter the position to move (row,col)")
        i = int(input("Row:"))
        j = int(input("Col:"))
        if board[i][j] == EMPTY:
          board = result(board, (i, j))
          print(numpy.array(board))
